# 06 — RAG Engine

**Module notebook — definitions only.**

Builds the retrieval-augmented chain used to chat with the transcript.

Depends on: `build_vector_store`, `load_vector_store`, `get_retriever` (from `05_vector_store.ipynb`) and `get_llm()` (from `00_llm_config.ipynb`).

> ## ⚠️ Legacy / not used by the live pipeline
>
> **the LangChain/Chroma RAG chain this notebook builds is no longer used by `09_agents.ipynb` or the LangGraph
> orchestrator (`11_orchestrator.ipynb`).** It was replaced by
> `08_llamaindex_retriever.ipynb`, which handles RAG retrieval for the
> live `rag_agent`. `content_agent`/`rag_agent` in `09_agents.ipynb` call
> `build_llama_index_bundle()` / `ask_llama_question()` from `08`, never
> anything from this notebook.
>
> This notebook is kept only because `main.ipynb`'s standalone
> `run_pipeline()` helper (a manual-testing convenience, separate from
> the LangGraph app) still uses it. If that helper is ever removed too,
> this notebook -- and the `langchain-chroma` / `chromadb` /
> `langchain-huggingface` dependencies it pulls in -- can be deleted
> entirely.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

RAG_SYSTEM_PROMPT_TEMPLATE = """You are an expert meeting assistant. Answer the user's question
based ONLY on the meeting transcript context provided below.

If the answer is not found in the context, say (in the required output language):
"I could not find this information in the meeting transcript."

Always be concise and precise. If quoting someone, mention it clearly.

{lang_instruction}

Context from meeting transcript:
{{context}}"""


In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [ ]:
def _build_chain_from_retriever(retriever, language: str = "english"):
    llm = get_llm()
    system_prompt = RAG_SYSTEM_PROMPT_TEMPLATE.format(
        lang_instruction=output_language_instruction(language)
    )
    prompt = ChatPromptTemplate.from_messages(
        [("system", system_prompt), ("human", "{question}")]
    )
    return (
        {
            "context": retriever | RunnableLambda(format_docs),
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )


def build_rag_chain(transcript: str, language: str = "english"):
    """Build a fresh vector store for this transcript and wire up the RAG chain."""
    vector_store = build_vector_store(transcript)
    retriever = get_retriever(vector_store, k=4)
    return _build_chain_from_retriever(retriever, language=language)


def load_rag_chain(language: str = "english"):
    """Reuse an already-persisted vector store (e.g. across sessions)."""
    vector_store = load_vector_store()
    retriever = get_retriever(vector_store)
    return _build_chain_from_retriever(retriever, language=language)


In [ ]:
def ask_question(rag_chain, question: str) -> str:
    print(f"Question: {question}")
    answer = rag_chain.invoke(question)
    print(f"Answer: {answer}")
    return answer
